In [ ]:
# ============================================================
#  CAS 3 : DÉTECTION EPI — YOLOv8n
#  Mémoire IPSSI 2026 — OZDEMIR Sedanur
# ============================================================

!pip install ultralytics -q

import os, glob, yaml, random, shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

print("✅ Installation OK")

# ============================================================
# ÉTAPE 1 — CHARGEMENT DU DATASET
# ============================================================

# Décompression du ZIP uploadé
import zipfile

ZIP_NAME = "PPE Identifier.v5i.yolov8.zip"
DATASET_PATH = "/content/ppe_dataset"

os.makedirs(DATASET_PATH, exist_ok=True)
with zipfile.ZipFile(ZIP_NAME, 'r') as z:
    z.extractall(DATASET_PATH)

print(f"✅ Dataset chargé : {DATASET_PATH}")

# Lecture du data.yaml
yaml_path = os.path.join(DATASET_PATH, "data.yaml")
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

CLASS_NAMES = cfg['names']
N_CLASSES   = cfg['nc']
print(f"   Classes ({N_CLASSES}) : {CLASS_NAMES}")

# ============================================================
# ÉTAPE 2 — STATISTIQUES RAPIDES
# ============================================================

splits = ['train', 'valid', 'test']
total_imgs, total_annot = 0, 0
class_counts = defaultdict(int)

for split in splits:
    imgs = glob.glob(f"{DATASET_PATH}/{split}/images/*.jpg") + \
           glob.glob(f"{DATASET_PATH}/{split}/images/*.png")
    lbls = glob.glob(f"{DATASET_PATH}/{split}/labels/*.txt")
    
    for lbl in lbls:
        with open(lbl) as f:
            for line in f:
                if line.strip():
                    class_counts[int(line.split()[0])] += 1
                    total_annot += 1
    
    total_imgs += len(imgs)
    print(f"   {split:6s} : {len(imgs):4d} images")

print(f"\n   Total : {total_imgs} images | {total_annot} annotations")
print(f"\n   Distribution par classe :")
for i, name in enumerate(CLASS_NAMES):
    print(f"   {name:25s} : {class_counts[i]:4d}")

# ============================================================
# ÉTAPE 3 — VISUALISATION EDA (simplifié)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Analyse Exploratoire — PPE Identifier Dataset\n'
             'Cas 3 : Détection EPI | Mémoire IPSSI 2026',
             fontsize=13, fontweight='bold')

colors = plt.cm.tab10(np.linspace(0, 1, N_CLASSES))

# Distribution des classes
ax = axes[0]
counts = [class_counts[i] for i in range(N_CLASSES)]
bars = ax.bar(CLASS_NAMES, counts, color=colors, edgecolor='white')
for bar, c in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(counts)*0.01,
            str(c), ha='center', fontweight='bold', fontsize=9)
ax.set_title('Distribution des classes', fontweight='bold')
ax.set_ylabel("Annotations")
ax.tick_params(axis='x', rotation=45)

# Camembert
ax = axes[1]
non_zero = [(CLASS_NAMES[i], counts[i]) for i in range(N_CLASSES) if counts[i] > 0]
labels_p, sizes_p = zip(*non_zero)
ax.pie(sizes_p, labels=labels_p, autopct='%1.1f%%',
       colors=colors[:len(non_zero)], startangle=90)
ax.set_title('Répartition globale', fontweight='bold')

# Images par split
ax = axes[2]
split_counts = []
for split in splits:
    imgs = glob.glob(f"{DATASET_PATH}/{split}/images/*.jpg") + \
           glob.glob(f"{DATASET_PATH}/{split}/images/*.png")
    split_counts.append(len(imgs))

bars2 = ax.bar(['Train', 'Valid', 'Test'], split_counts,
               color=['#3498db', '#e67e22', '#2ecc71'],
               edgecolor='white', linewidth=1.5)
for bar, c in zip(bars2, split_counts):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(split_counts)*0.01,
            str(c), ha='center', fontweight='bold')
ax.set_title("Images par split", fontweight='bold')
ax.set_ylabel("Nombre d'images")

plt.tight_layout()
plt.savefig('eda_ppe.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA sauvegardée")

# ============================================================
# ÉTAPE 4 — EXEMPLES D'IMAGES ANNOTÉES
# ============================================================

train_imgs = glob.glob(f"{DATASET_PATH}/train/images/*.jpg") + \
             glob.glob(f"{DATASET_PATH}/train/images/*.png")
selected = random.sample(train_imgs, min(8, len(train_imgs)))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for idx, img_path in enumerate(selected):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    lbl_path = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    n_det = 0
    
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id = int(parts[0])
                    cx, cy, bw, bh = map(float, parts[1:])
                    x1 = int((cx - bw/2) * w)
                    y1 = int((cy - bh/2) * h)
                    x2 = int((cx + bw/2) * w)
                    y2 = int((cy + bh/2) * h)
                    c = colors[cls_id % N_CLASSES]
                    color_cv = (int(c[0]*255), int(c[1]*255), int(c[2]*255))
                    cv2.rectangle(img_rgb, (x1,y1), (x2,y2), color_cv, 2)
                    label = CLASS_NAMES[cls_id] if cls_id < N_CLASSES else str(cls_id)
                    cv2.putText(img_rgb, label, (x1, max(y1-5,10)),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_cv, 2)
                    n_det += 1
    
    axes[idx].imshow(img_rgb)
    axes[idx].axis('off')
    axes[idx].set_title(f"{n_det} annotation(s)", fontsize=9)

legend_patches = [mpatches.Patch(color=colors[i], label=CLASS_NAMES[i])
                  for i in range(N_CLASSES)]
fig.legend(handles=legend_patches, loc='lower center',
           ncol=N_CLASSES, fontsize=9, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Exemples annotés — PPE Identifier\n'
             'Mémoire IPSSI 2026 — OZDEMIR Sedanur',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exemples_EPI.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Exemples sauvegardés")

# ============================================================
# ÉTAPE 5 — ENTRAÎNEMENT YOLOV8n
# ============================================================

# Correction chemins data.yaml
with open(yaml_path, 'r') as f:
    content = f.read()

yaml_fixed = '/content/ppe_data.yaml'
with open(yaml_fixed, 'w') as f:
    f.write(f"path: {DATASET_PATH}\n"
            f"train: train/images\n"
            f"val: valid/images\n"
            f"test: test/images\n"
            f"nc: {N_CLASSES}\n"
            f"names: {CLASS_NAMES}\n")

model = YOLO('yolov8n.pt')
print("✅ Modèle YOLOv8n chargé\n⏳ Entraînement...")

results = model.train(
    data=yaml_fixed,
    epochs=100,
    imgsz=640,
    batch=16,
    project='/content/ppe_runs',
    name='yolov8n_ppe',
    patience=30,
    optimizer='AdamW',
    lr0=0.001,
    verbose=False
)

print("✅ Entraînement terminé !")

# ============================================================
# ÉTAPE 6 — ÉVALUATION SUR LE JEU DE TEST
# ============================================================

best_model = YOLO('/content/ppe_runs/yolov8n_ppe/weights/best.pt')

test_res = best_model.val(
    data=yaml_fixed,
    split='test',
    imgsz=640,
    verbose=True
)

print("\n" + "="*50)
print("  RÉSULTATS — JEU DE TEST")
print("="*50)
print(f"  mAP@0.5       : {test_res.box.map50:.4f}")
print(f"  mAP@0.5:0.95  : {test_res.box.map:.4f}")
print(f"  Precision     : {test_res.box.mp:.4f}")
print(f"  Recall        : {test_res.box.mr:.4f}")
print("="*50)

# Par classe
print(f"\n{'Classe':<20} {'AP@0.5':>8} {'Precision':>10} {'Recall':>8}")
print("-"*48)
for i, name in enumerate(CLASS_NAMES):
    if i < len(test_res.box.ap50):
        print(f"  {name:<18} "
              f"{test_res.box.ap50[i]:>8.4f} "
              f"{test_res.box.p[i]:>10.4f} "
              f"{test_res.box.r[i]:>8.4f}")

# ============================================================
# ÉTAPE 7 — IMPACT MÉTIER SÉCURITÉ
# ============================================================

print("\n" + "="*60)
print("  ANALYSE IMPACT MÉTIER — SÉCURITÉ")
print("="*60)

N_WORKERS        = 150
WORKING_DAYS     = 220
NON_CONF_RATE    = 0.15
ACCIDENT_RATE    = 8.5   # pour 1000 travailleurs
COST_ACCIDENT    = 35000
DETECT_BASELINE  = 0.40
DETECT_AI        = test_res.box.mr

# Calculs
nc_total     = N_WORKERS * NON_CONF_RATE * WORKING_DAYS
nc_missed_bl = nc_total * (1 - DETECT_BASELINE)
nc_missed_ai = nc_total * (1 - DETECT_AI)

acc_baseline = N_WORKERS * ACCIDENT_RATE / 1000
acc_ai       = acc_baseline * (1 - DETECT_AI * 0.6)
acc_avoided  = acc_baseline - acc_ai
cost_saved   = acc_avoided * COST_ACCIDENT

conf_baseline = DETECT_BASELINE * 100
conf_ai       = test_res.box.mp * 100

print(f"\n{'Indicateur':<35} {'Baseline':>10} {'IA':>10} {'Gain':>10}")
print("-"*67)
print(f"{'Conformité EPI (%)':<35} {conf_baseline:>9.1f}% {conf_ai:>9.1f}% {conf_ai-conf_baseline:>+9.1f}%")
print(f"{'Non-conform. manquées/an':<35} {nc_missed_bl:>10.0f} {nc_missed_ai:>10.0f} {nc_missed_ai-nc_missed_bl:>+10.0f}")
print(f"{'Accidents estimés/an':<35} {acc_baseline:>10.1f} {acc_ai:>10.1f} {-acc_avoided:>+10.1f}")
print(f"{'Économie annuelle (€)':<35} {'':>10} {'':>10} {cost_saved:>+10,.0f}")
print("="*60)

# Graphique impact
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Impact métier — Sécurité & Conformité EPI\n'
             'Cas 3 | Mémoire IPSSI 2026 — OZDEMIR Sedanur',
             fontsize=13, fontweight='bold')

# Conformité
ax = axes[0]
cats = ['Système\nclassique', 'YOLOv8n\n(IA)']
vals = [conf_baseline, conf_ai]
bars = ax.bar(cats, vals, color=['#e74c3c', '#2ecc71'],
              edgecolor='white', linewidth=2, width=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f'{v:.1f}%', ha='center', fontsize=14, fontweight='bold')
ax.set_ylim(0, 110)
ax.set_ylabel('Taux de conformité EPI (%)')
ax.set_title('Conformité EPI', fontweight='bold')
ax.text(0.5, 0.45, f'+{conf_ai-conf_baseline:.1f} pts',
        transform=ax.transAxes, ha='center',
        color='#27ae60', fontsize=14, fontweight='bold')

# Tableau synthèse
ax = axes[1]
ax.axis('off')
tdata = [
    ['Indicateur', 'Valeur'],
    ['mAP@0.5', f"{test_res.box.map50:.3f}"],
    ['Recall', f"{DETECT_AI*100:.1f}%"],
    ['Precision', f"{test_res.box.mp*100:.1f}%"],
    ['Gain conformité', f'+{conf_ai-conf_baseline:.1f} pts'],
    ['Accidents évités/an', f'{acc_avoided:.1f}'],
    ['Économie annuelle', f'{cost_saved:,.0f} €'],
]
table = ax.table(cellText=tdata[1:], colLabels=tdata[0],
                 cellLoc='center', loc='center', bbox=[0,0,1,1])
table.auto_set_font_size(False)
table.set_fontsize(11)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#d5f5e3')
    cell.set_edgecolor('white')
ax.set_title('Synthèse impact sécurité', fontweight='bold')

plt.tight_layout()
plt.savefig('impact_securite.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Impact métier sauvegardé")

# Sauvegarde modèle
shutil.copy('/content/ppe_runs/yolov8n_ppe/weights/best.pt',
            '/content/yolov8n_ppe_best.pt')
print("✅ Modèle sauvegardé")

In [ ]:
# ============================================================
#  CAS 3 — PARTIE 2 : ANALYSE DE POSTURE (MediaPipe)
#  Détection postures à risque — Mémoire IPSSI 2026
# ============================================================

!pip install mediapipe -q

import cv2
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from google.colab.patches import cv2_imshow
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print("✅ MediaPipe installé")

# ============================================================
# CONFIGURATION MEDIAPIPE
# ============================================================

mp_pose    = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_styles  = mp.solutions.drawing_styles

# Seuils de risque (angles en degrés)
SEUIL_FLEXION_ROUGE  = 45   # flexion > 45° = risque élevé
SEUIL_FLEXION_ORANGE = 20   # flexion > 20° = risque modéré

# ============================================================
# FONCTIONS UTILITAIRES
# ============================================================

def calculate_angle(a, b, c):
    """
    Calcule l'angle en degrés entre 3 points (a-b-c)
    b = sommet de l'angle
    """
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    
    ba = a - b
    bc = c - b
    
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    angle  = np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))
    return angle


def get_risk_level(angle_flexion, angle_genoux):
    """
    Détermine le niveau de risque selon les angles
    Retourne : (niveau, couleur, message)
    """
    if angle_flexion > SEUIL_FLEXION_ROUGE:
        return "RISQUE ÉLEVÉ", (0, 0, 255), "Flexion lombaire excessive !"
    elif angle_flexion > SEUIL_FLEXION_ORANGE:
        return "RISQUE MODÉRÉ", (0, 165, 255), "Posture à surveiller"
    else:
        return "CONFORME", (0, 200, 0), "Posture correcte"


def draw_angle_arc(img, point, angle, color, radius=30):
    """Dessine un arc indiquant l'angle mesuré"""
    h, w = img.shape[:2]
    cx = int(point[0] * w)
    cy = int(point[1] * h)
    cv2.ellipse(img, (cx, cy), (radius, radius), 0, 0, int(angle),
                color, 2)
    cv2.putText(img, f"{angle:.0f}°", (cx + radius + 5, cy),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)


def analyze_frame(frame, pose):
    """
    Analyse une frame et retourne les résultats
    """
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(img_rgb)
    
    output = frame.copy()
    analysis = {
        'detected': False,
        'angle_dos': 0,
        'angle_genoux': 0,
        'risk_level': 'INCONNU',
        'risk_color': (128, 128, 128),
        'message': 'Aucune personne détectée'
    }
    
    if not results.pose_landmarks:
        return output, analysis
    
    lm = results.pose_landmarks.landmark
    h, w = frame.shape[:2]
    
    # Points clés du squelette
    # Épaule gauche
    epaule_g = [lm[mp_pose.PoseLandmark.LEFT_SHOULDER].x,
                lm[mp_pose.PoseLandmark.LEFT_SHOULDER].y]
    # Hanche gauche
    hanche_g = [lm[mp_pose.PoseLandmark.LEFT_HIP].x,
                lm[mp_pose.PoseLandmark.LEFT_HIP].y]
    # Genou gauche
    genou_g  = [lm[mp_pose.PoseLandmark.LEFT_KNEE].x,
                lm[mp_pose.PoseLandmark.LEFT_KNEE].y]
    # Cheville gauche
    cheville_g = [lm[mp_pose.PoseLandmark.LEFT_ANKLE].x,
                  lm[mp_pose.PoseLandmark.LEFT_ANKLE].y]
    
    # Point de référence vertical (au-dessus de la hanche)
    vertical_ref = [hanche_g[0], hanche_g[1] - 0.3]
    
    # Calcul des angles
    angle_dos    = calculate_angle(epaule_g, hanche_g, vertical_ref)
    angle_genoux = calculate_angle(hanche_g, genou_g, cheville_g)
    
    # Niveau de risque
    risk_level, risk_color, message = get_risk_level(
        angle_dos, angle_genoux
    )
    
    # Dessin du squelette
    mp_drawing.draw_landmarks(
        output,
        results.pose_landmarks,
        mp_pose.POSE_CONNECTIONS,
        landmark_drawing_spec=mp_drawing.DrawingSpec(
            color=risk_color, thickness=3, circle_radius=4
        ),
        connection_drawing_spec=mp_drawing.DrawingSpec(
            color=risk_color, thickness=2
        )
    )
    
    # Dessin des angles
    draw_angle_arc(output, hanche_g, angle_dos, risk_color)
    draw_angle_arc(output, genou_g, angle_genoux, (255, 200, 0))
    
    # Bandeau de statut
    overlay = output.copy()
    cv2.rectangle(overlay, (0, 0), (w, 80), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, output, 0.4, 0, output)
    
    # Texte statut
    cv2.putText(output, risk_level, (10, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, risk_color, 3)
    cv2.putText(output, message, (10, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    # Métriques
    cv2.putText(output, f"Flexion dos: {angle_dos:.0f}°",
                (w - 250, 35), cv2.FONT_HERSHEY_SIMPLEX,
                0.7, risk_color, 2)
    cv2.putText(output, f"Angle genoux: {angle_genoux:.0f}°",
                (w - 250, 65), cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (255, 200, 0), 2)
    
    analysis.update({
        'detected': True,
        'angle_dos': angle_dos,
        'angle_genoux': angle_genoux,
        'risk_level': risk_level,
        'risk_color': risk_color,
        'message': message
    })
    
    return output, analysis

# ============================================================
# TRAITEMENT DE LA VIDÉO
# ============================================================

# Upload de la vidéo
print("📁 Uploadez votre vidéo (MP4) :")
uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print(f"✅ Vidéo chargée : {VIDEO_PATH}")

# Traitement
cap = cv2.VideoCapture(VIDEO_PATH)
fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"   Résolution : {width}x{height} | FPS : {fps:.1f} | "
      f"Frames : {total_frames}")

# Sortie vidéo annotée
OUTPUT_VIDEO = '/content/posture_analysis_output.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

# Historique pour graphiques
history = {
    'frame': [],
    'angle_dos': [],
    'angle_genoux': [],
    'risk_level': []
}

# Frames pour visualisation
sample_frames = []
sample_interval = max(1, total_frames // 6)

with mp_pose.Pose(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as pose:
    
    frame_idx = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        output_frame, analysis = analyze_frame(frame, pose)
        out.write(output_frame)
        
        if analysis['detected']:
            history['frame'].append(frame_idx)
            history['angle_dos'].append(analysis['angle_dos'])
            history['angle_genoux'].append(analysis['angle_genoux'])
            history['risk_level'].append(analysis['risk_level'])
        
        # Sauvegarde frames exemples
        if frame_idx % sample_interval == 0:
            sample_frames.append((frame_idx, output_frame.copy()))
        
        frame_idx += 1
        
        if frame_idx % 50 == 0:
            print(f"   Traitement : {frame_idx}/{total_frames} frames")

cap.release()
out.release()
print(f"\n✅ Vidéo annotée sauvegardée : {OUTPUT_VIDEO}")

# ============================================================
# VISUALISATION DES FRAMES EXEMPLES
# ============================================================

if sample_frames:
    n = min(6, len(sample_frames))
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for idx, (frame_num, frame) in enumerate(sample_frames[:n]):
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(frame_rgb)
        axes[idx].axis('off')
        axes[idx].set_title(f"Frame {frame_num}", fontsize=9)
    
    plt.suptitle('Analyse de posture — MediaPipe\n'
                 'Cas 3 : Détection comportements à risque | '
                 'Mémoire IPSSI 2026 — OZDEMIR Sedanur',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('frames_posture.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Frames exemples sauvegardées")

# ============================================================
# GRAPHIQUES D'ANALYSE TEMPORELLE
# ============================================================

if history['frame']:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Analyse temporelle des postures — MediaPipe\n'
                 'Mémoire IPSSI 2026 — OZDEMIR Sedanur',
                 fontsize=13, fontweight='bold')
    
    frames = history['frame']
    angles_dos = history['angle_dos']
    angles_gen = history['angle_genoux']
    
    # Évolution angle dos
    ax = axes[0, 0]
    ax.plot(frames, angles_dos, color='#e74c3c', linewidth=1.5)
    ax.axhline(SEUIL_FLEXION_ROUGE, color='red',
               linestyle='--', linewidth=2,
               label=f'Seuil risque élevé ({SEUIL_FLEXION_ROUGE}°)')
    ax.axhline(SEUIL_FLEXION_ORANGE, color='orange',
               linestyle='--', linewidth=2,
               label=f'Seuil risque modéré ({SEUIL_FLEXION_ORANGE}°)')
    ax.fill_between(frames, angles_dos, SEUIL_FLEXION_ROUGE,
                    where=[a > SEUIL_FLEXION_ROUGE for a in angles_dos],
                    alpha=0.3, color='red', label='Zone risque élevé')
    ax.set_title('Évolution angle de flexion du dos', fontweight='bold')
    ax.set_xlabel('Frame')
    ax.set_ylabel('Angle (°)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Évolution angle genoux
    ax = axes[0, 1]
    ax.plot(frames, angles_gen, color='#3498db', linewidth=1.5)
    ax.set_title('Évolution angle des genoux', fontweight='bold')
    ax.set_xlabel('Frame')
    ax.set_ylabel('Angle (°)')
    ax.grid(True, alpha=0.3)
    
    # Distribution des niveaux de risque
    ax = axes[1, 0]
    risk_counts = {
        'CONFORME': history['risk_level'].count('CONFORME'),
        'RISQUE MODÉRÉ': history['risk_level'].count('RISQUE MODÉRÉ'),
        'RISQUE ÉLEVÉ': history['risk_level'].count('RISQUE ÉLEVÉ')
    }
    risk_colors = ['#2ecc71', '#f39c12', '#e74c3c']
    non_zero_risks = {k: v for k, v in risk_counts.items() if v > 0}
    
    if non_zero_risks:
        bars = ax.bar(non_zero_risks.keys(),
                      non_zero_risks.values(),
                      color=risk_colors[:len(non_zero_risks)],
                      edgecolor='white', linewidth=1.5)
        for bar, v in zip(bars, non_zero_risks.values()):
            pct = v / len(history['risk_level']) * 100
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(non_zero_risks.values())*0.01,
                    f'{v}\n({pct:.1f}%)', ha='center',
                    fontweight='bold', fontsize=9)
    ax.set_title('Distribution des niveaux de risque', fontweight='bold')
    ax.set_ylabel('Nombre de frames')
    
    # Tableau récapitulatif
    ax = axes[1, 1]
    ax.axis('off')
    
    total_detected = len(history['frame'])
    pct_conforme = risk_counts['CONFORME'] / max(total_detected, 1) * 100
    pct_modere   = risk_counts['RISQUE MODÉRÉ'] / max(total_detected, 1) * 100
    pct_eleve    = risk_counts['RISQUE ÉLEVÉ'] / max(total_detected, 1) * 100
    
    tdata = [
        ['Métrique', 'Valeur'],
        ['Frames analysées', str(total_detected)],
        ['Angle dos moyen', f"{np.mean(angles_dos):.1f}°"],
        ['Angle dos max', f"{max(angles_dos):.1f}°"],
        ['Postures conformes', f"{pct_conforme:.1f}%"],
        ['Risque modéré', f"{pct_modere:.1f}%"],
        ['Risque élevé', f"{pct_eleve:.1f}%"],
        ['Seuil risque élevé', f"> {SEUIL_FLEXION_ROUGE}°"],
    ]
    
    table = ax.table(cellText=tdata[1:], colLabels=tdata[0],
                     cellLoc='center', loc='center', bbox=[0,0,1,1])
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    for (r, c), cell in table.get_celld().items():
        if r == 0:
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold')
        elif r % 2 == 0:
            cell.set_facecolor('#d5f5e3')
        cell.set_edgecolor('white')
    ax.set_title('Récapitulatif analyse posturale', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('analyse_posture_temporelle.png', dpi=150,
                bbox_inches='tight')
    plt.show()
    print("✅ Analyse temporelle sauvegardée")

# ============================================================
# RÉCAPITULATIF FINAL
# ============================================================

print("\n" + "="*55)
print("  RÉCAPITULATIF — MODULE MEDIAPIPE")
print("="*55)
if history['frame']:
    total = len(history['frame'])
    print(f"  Frames analysées    : {total}")
    print(f"  Angle dos moyen     : {np.mean(angles_dos):.1f}°")
    print(f"  Postures conformes  : {pct_conforme:.1f}%")
    print(f"  Risque modéré       : {pct_modere:.1f}%")
    print(f"  Risque élevé        : {pct_eleve:.1f}%")
print("="*55)

# Téléchargement
files.download(OUTPUT_VIDEO)
files.download('frames_posture.png')
files.download('analyse_posture_temporelle.png')
print("✅ Fichiers téléchargés !")

ource
Lien
Pourquoi
Pexels (gratuit)
pexels.com/video/3249063
Ouvriers en entrepôt, postures variées
Pexels (gratuit)
pexels.com/video/4065977
Travailleur portant des charges
Pixabay (gratuit)
pixabay.com/videos/worker-factory
Usine, mouvements réalistes
💡 Conseil : Choisissez une vidéo avec une seule personne visible pour commencer, MediaPipe fonctionne mieux sur une personne à la fois avec ce code.